In [3]:
import cv2
import pygame
from time import sleep
from cvzone.HandTrackingModule import HandDetector
from pynput.keyboard import Key, Controller
import pyautogui
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

last_click_time = None
double_tap_threshold = 0.5

def on_button_click(b):
    global last_click_time
    current_time = time.time()

    if last_click_time:
        click_interval = current_time - last_click_time
        if click_interval < double_tap_threshold:
            handle_double_tap()
        else:
            print("Single tap detected.")
    else:
        print("First tap detected.")

    last_click_time = current_time

def handle_double_tap():
    clear_output(wait=True)  # Clear previous output
    print("Double-tap detected!")

button = widgets.Button(description="Click Me")
button.on_click(on_button_click)
display(button)


# Initialize video capture
cap = cv2.VideoCapture(0)
cap.set(3, 1200)  # Width
cap.set(4, 800)  # Height
#hand dictetcor lagatya
detector = HandDetector(detectionCon=0.8)
keyboard = Controller()

# keyboard keys lagai
class Keys:
    def __init__(self, pos, text, size=[80, 80]):
        self.pos = pos
        self.text = text
        self.size = size

    def draw(self, img):
        x, y = self.pos
        w, h = self.size
        cv2.rectangle(img, self.pos, (x + w, y + h), (204, 0, 0), cv2.FILLED)
        cv2.putText(img, self.text, (self.pos[0] + 20, self.pos[1] + 60), cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 2)
        return img

keyboard_layout = [
    ['1', '2', '3', '4', '5', '6', '7', '8', '9', '0'],
    ['Q', 'W', 'E', 'R', 'T', 'Y', 'U', 'I', 'O', 'P'],
    ['A', 'S', 'D', 'F', 'G', 'H', 'J', 'K', 'L', ','],
    ['Z', 'X', 'C', 'V', 'B', 'N', 'M', '.', '?', ' ']
]

key_lst = []
for row_idx, row in enumerate(keyboard_layout):
    for col_idx, char in enumerate(row):
        key_lst.append(Keys((80 * col_idx + col_idx * 10 + 200, 50 + row_idx * 90), char))

output_txt = ""

def close_keyboard():
    pyautogui.press('esc')

print("You have 5 seconds to open the keyboard...")
time.sleep(5)

while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    hands, img = detector.findHands(img, flipType=False)

    for key in key_lst:
        key.draw(img)

    cv2.rectangle(img, (1010, 410), (1090, 500), (234, 0, 0), cv2.FILLED)
    cv2.putText(img, "<-", (1015, 470), cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 2)

    if hands:
        hand = hands[0]
        lmList = hand["lmList"]
        bbox = hand["bbox"]
        length, _, _ = detector.findDistance(lmList[8][0:2], lmList[4][0:2], img)

        for key in key_lst:
            x, y = key.pos
            w, h = key.size
            if x < lmList[8][0] < x + w and y < lmList[8][1] < y + h:
                cv2.rectangle(img, key.pos, (x + w, y + h), (255, 153, 51), cv2.FILLED)  # Highlighted color
                cv2.putText(img, key.text, (key.pos[0] + 10, key.pos[1] + 60), cv2.FONT_HERSHEY_PLAIN, 5, (255, 255, 255), 2)
                length, _, _ = detector.findDistance(lmList[8][0:2], lmList[4][0:2], img)

                if length < 40:
                    keyboard.press(key.text)
                    cv2.rectangle(img, key.pos, (x + w, y + h), (0, 153, 0), cv2.FILLED)  # Pressed color
                    cv2.putText(img, key.text, (key.pos[0] + 10, key.pos[1] + 40), cv2.FONT_HERSHEY_PLAIN, 5, (255, 255, 255), 2)
                    output_txt += key.text
                    hover_sound.play()
                    sleep(0.25)

        # backspace
        if 1010 < lmList[8][0] < 1090 and 410 < lmList[8][1] < 500:
            cv2.rectangle(img, (1010, 410), (1090, 500), (255, 153, 51), cv2.FILLED)
            cv2.putText(img, "<-", (1015, 470), cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 2)

            if length < 40:
                keyboard.press(Key.backspace)
                if output_txt:
                    output_txt = output_txt[:-1]
                    hover_sound.play()
                    sleep(0.2)

                    
    cv2.rectangle(img, (200, 410), (1000, 500), (234, 0, 0), cv2.FILLED)
    cv2.putText(img, output_txt, (210, 470), cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 2)

    cv2.imshow("Hand Tracking", img)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC key to exit
        break

cap.release()
cv2.destroyAllWindows()
close_keyboard()


Button(description='Click Me', style=ButtonStyle())

You have 5 seconds to open the keyboard...
First tap detected.
